# Episode 006 — Bitcoin Baseline Stability

This CodeLab audits the saved validation evidence for `BITCOIN_i` H1 direction. It reads provenance, fold membership, predictions, and metrics from the frozen EP006 run. It does not load raw price bars, fit estimators, tune parameters, or score the locked outer test.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import pandas as pd

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / 'configs/experiments/ep006_btc_stability.toml').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this CodeLab from inside the project checkout.')
RUN_DIR = Path(os.environ.get('EP006_RUN_DIR', PROJECT_ROOT / '.local/ep006-bitcoin-stability-run')).resolve()
manifest = json.loads((RUN_DIR / 'run_manifest.json').read_text(encoding='utf-8'))
metrics = json.loads((RUN_DIR / 'metrics.json').read_text(encoding='utf-8'))
report = json.loads((RUN_DIR / 'report.json').read_text(encoding='utf-8'))
predictions = pd.read_csv(RUN_DIR / 'predictions.csv')
membership = pd.read_csv(RUN_DIR / 'window_membership.csv')
print(f"Run: {manifest['experiment_id']} | symbol: {manifest['source']['symbol']}")
print(f"Raw manifest SHA256: {manifest['source']['raw_manifest_sha256']}")
print(f"Locked test evaluated: {manifest['locked_test_evaluated']}")

In [ ]:
for name, expected in manifest['artifacts'].items():
    digest = hashlib.sha256((RUN_DIR / name).read_bytes()).hexdigest()
    assert digest == expected['sha256'], f'artifact hash mismatch: {name}'
for relative, expected in manifest['implementation_sha256'].items():
    digest = hashlib.sha256((PROJECT_ROOT / relative).read_bytes()).hexdigest()
    assert digest == expected, f'execution source hash mismatch: {relative}'

assert manifest['locked_test_evaluated'] is False
assert manifest['locked_test_fit_or_scored'] is False
assert report['locked_test']['status'] == 'locked'
assert report['locked_test']['evaluated'] is False
assert 'metrics' not in report['locked_test']
folds = {fold['window_id']: fold for fold in manifest['folds']}
assert set(folds) == {'V1', 'V2', 'V3', 'V4'}
test_start = pd.Timestamp(manifest['outer_split']['test_start_utc'])
predictions['decision_timestamp'] = pd.to_datetime(predictions['decision_timestamp'], utc=True)
predictions['target_timestamp'] = pd.to_datetime(predictions['target_timestamp'], utc=True)
assert not predictions.duplicated(['window_id', 'decision_timestamp']).any()
assert predictions['decision_timestamp'].is_unique
assert (predictions['decision_timestamp'] < test_start).all()
assert not (membership['partition_role'] == 'locked_test').any()
assert not (pd.to_datetime(membership['decision_timestamp'], utc=True) >= test_start).any()

for window_id, fold in folds.items():
    start = pd.Timestamp(fold['start_utc'])
    end = pd.Timestamp(fold['end_utc_exclusive'])
    rows = membership.loc[membership['window_id'] == window_id]
    train = rows.loc[rows['partition_role'] == 'expanding_train']
    candidates = rows.loc[rows['partition_role'] == 'validation_candidate']
    train_decision = pd.to_datetime(train['decision_timestamp'], utc=True)
    train_target = pd.to_datetime(train['target_timestamp'], utc=True)
    candidate_decision = pd.to_datetime(candidates['decision_timestamp'], utc=True)
    candidate_target = pd.to_datetime(candidates['target_timestamp'], utc=True)
    eligible = candidates.loc[candidate_target < end]
    scored = candidates.loc[candidates['scored'].astype(bool)]
    assert len(train) == fold['expanding_train_rows']
    assert len(candidates) == fold['candidate_rows']
    assert len(eligible) == fold['scored_validation_rows']
    assert len(scored) == fold['scored_validation_rows']
    assert (train_decision < start).all() and (train_target < start).all()
    assert (candidate_decision >= start).all() and (candidate_decision < end).all()
    assert (candidate_target < test_start).all()
    assert (pd.to_datetime(eligible['target_timestamp'], utc=True) < end).all()
    eligible_keys = set(zip(pd.to_datetime(eligible['decision_timestamp'], utc=True), pd.to_datetime(eligible['target_timestamp'], utc=True)))
    scored_keys = set(zip(pd.to_datetime(scored['decision_timestamp'], utc=True), pd.to_datetime(scored['target_timestamp'], utc=True)))
    assert scored_keys == eligible_keys, f'scored membership differs from eligible rows for {window_id}'
    pred = predictions.loc[predictions['window_id'] == window_id]
    pred_keys = set(zip(pred['decision_timestamp'], pred['target_timestamp']))
    assert pred_keys == scored_keys, f'scored membership differs from predictions for {window_id}'
assert len(predictions) == sum(fold['scored_validation_rows'] for fold in folds.values())
print(f"Validated {len(predictions):,} validation predictions, fold purges, and locked-test separation.")

In [ ]:
baseline_names = report['baseline_names']

def audit_metrics(actual, predicted):
    labels = ('DOWN', 'UP')
    actual = pd.Series(actual, dtype='string')
    predicted = pd.Series(predicted, dtype='string')
    tn = int(((actual == labels[0]) & (predicted == labels[0])).sum())
    fp = int(((actual == labels[0]) & (predicted == labels[1])).sum())
    fn = int(((actual == labels[1]) & (predicted == labels[0])).sum())
    tp = int(((actual == labels[1]) & (predicted == labels[1])).sum())
    accuracy = (tn + tp) / len(actual)
    down_recall = tn / (tn + fp) if tn + fp else 0.0
    up_recall = tp / (tp + fn) if tp + fn else 0.0
    denominator = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / denominator if denominator else 0.0
    return {
        'rows': len(actual), 'accuracy': accuracy,
        'balanced_accuracy': (down_recall + up_recall) / 2,
        'mcc': mcc, 'confusion_matrix': {'labels': list(labels), 'values': [[tn, fp], [fn, tp]]},
    }

for window_id in ('V1', 'V2', 'V3', 'V4'):
    scored = predictions.loc[predictions['window_id'] == window_id]
    for name in baseline_names:
        calculated = audit_metrics(scored['actual_direction'], scored[name])
        stored = metrics['per_window'][window_id][name]
        assert calculated['rows'] == stored['rows']
        assert calculated['confusion_matrix'] == stored['confusion_matrix']
        for key in ('accuracy', 'balanced_accuracy', 'mcc'):
            assert math.isclose(calculated[key], stored[key], rel_tol=1e-12, abs_tol=1e-12)
for name in baseline_names:
    calculated = audit_metrics(predictions['actual_direction'], predictions[name])
    stored = metrics['pooled'][name]
    assert calculated['rows'] == stored['rows']
    assert calculated['confusion_matrix'] == stored['confusion_matrix']
    for key in ('accuracy', 'balanced_accuracy', 'mcc'):
        assert math.isclose(calculated[key], stored[key], rel_tol=1e-12, abs_tol=1e-12)
print('All four per-window metric sets and pooled metrics reconcile to saved predictions.')

In [ ]:
rows = []
for window_id in ('V1', 'V2', 'V3', 'V4', 'pooled'):
    values = metrics['pooled'] if window_id == 'pooled' else metrics['per_window'][window_id]
    row = {'window': window_id, 'rows': values[baseline_names[0]]['rows']}
    row.update({name: values[name]['accuracy'] for name in baseline_names})
    rows.append(row)
accuracy_table = pd.DataFrame(rows).set_index('window')
accuracy_table

## Reading the results

The four windows are sequential validation periods with expanding training folds. The pooled summary uses only the validation predictions present in this local run. The report contains classification metrics; it does not estimate transaction costs, execution quality, drawdown, or profitability. Do not use these validation results as evidence from the locked test.